In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import argparse
import numpy as np
from model import HIPT_CLIP_Model
from Dataset import TestDataset
import sys
import pandas as pd
from config import CFG
import torch.nn.functional as F

## demo run

# BRCA benchmark

## load pretrained model

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')

#model = HIPT_CLIP_Model().to(device)
#pretrained path
pretrained_weights_path = "/sibcb1/chenluonanlab8/zoujiawei/Final_version_with_foundation_model/model/checkpoint/brca/model.pt"
pretrained_state_dict = torch.load(pretrained_weights_path, map_location=torch.device('cpu'))

# Initialize the CLIP model
hipt_model = HIPT_CLIP_Model().to(device)
hipt_model.load_state_dict(pretrained_state_dict)
hipt_model.eval()
model = hipt_model


Take key teacher in provided checkpoint dict
Pretrained weights found at /sibcb1/chenluonanlab8/zoujiawei/Final_version_with_foundation_model/img_encoder/HIPT_4K/Checkpoints/vit256_small_dino.pth and loaded with msg: _IncompatibleKeys(missing_keys=[], unexpected_keys=['head.mlp.0.weight', 'head.mlp.0.bias', 'head.mlp.2.weight', 'head.mlp.2.bias', 'head.mlp.4.weight', 'head.mlp.4.bias', 'head.last_layer.weight_g', 'head.last_layer.weight_v'])
model_HPIT_load_success!


RuntimeError: Error(s) in loading state_dict for HIPT_CLIP_Model:
	size mismatch for spot_projection.projection.weight: copying a param with shape torch.Size([384, 2000]) from checkpoint, the shape in current model is torch.Size([384, 512]).

## load test datasets

In [3]:
from Dataset_Test import ImageDataset, ExpressionDataset

In [4]:
## test dataset paths
#
def build_paths(base_path, ids, extension, remove_prefix=None):
    """Builds file paths, optionally removing a prefix from IDs.

    Args:
        base_path: Base directory path (should end with '/').
        ids: List of IDs (strings).
        extension: File extension (including the leading dot '.').
        remove_prefix: Prefix to remove from IDs (string, or None if no removal).

    Returns:
        List of file paths.  Returns an empty list if input is invalid or an error occurs.
    """
    try:

        cleaned_ids = [id.replace(remove_prefix, "") if remove_prefix else id for id in ids]

        return [base_path + id + extension for id in cleaned_ids]

    except Exception as e:
        print(f"An error occurred: {e}")
        return []
ids = "H2"
image_paths = "/sibcb1/chenluonanlab8/zoujiawei/Final_version_with_foundation_model/data/BRCA/image_rename/standardized_"+ ids+ ".jpg"
spatial_pos_paths = "/sibcb1/chenluonanlab8/zoujiawei/Final_version_with_foundation_model/data/BRCA/st/spot_"+ ids+ ".csv"

ori_count_path = "/sibcb1/chenluonanlab8/zoujiawei/Final_version_with_foundation_model/data/BRCA/st/oricount_"+ ids+ ".csv"

In [5]:
image_paths

'/sibcb1/chenluonanlab8/zoujiawei/Final_version_with_foundation_model/data/BRCA/image_rename/standardized_H2.jpg'

In [6]:
    ## Ref adata of gene expression 
    import scanpy as sc
    sample_data_path = '/sibcb1/chenluonanlab8/zoujiawei/Final_version_with_foundation_model/scGPT/BRCA_data_scGPT.h5ad'
    adata = sc.read_h5ad(sample_data_path)

    # 2. Define your sample ID and batch size (and other parameters)
    sample_id = "H2"  #  example, replace if needed

In [7]:
adata_subset = adata[adata.obs['sample'] == sample_id, :].copy()


In [8]:
adata_subset.obs


,sample,image_path,cell_type
H2_cell_1,H2,/sibcb1/chenluonanlab8/zoujiawei/Final_version...,5
H2_cell_2,H2,/sibcb1/chenluonanlab8/zoujiawei/Final_version...,5
H2_cell_3,H2,/sibcb1/chenluonanlab8/zoujiawei/Final_version...,5
H2_cell_4,H2,/sibcb1/chenluonanlab8/zoujiawei/Final_version...,5
H2_cell_5,H2,/sibcb1/chenluonanlab8/zoujiawei/Final_version...,5
...,...,...,...
H2_cell_599,H2,/sibcb1/chenluonanlab8/zoujiawei/Final_version...,5
H2_cell_600,H2,/sibcb1/chenluonanlab8/zoujiawei/Final_version...,5
H2_cell_601,H2,/sibcb1/chenluonanlab8/zoujiawei/Final_version...,5
H2_cell_602,H2,/sibcb1/chenluonanlab8/zoujiawei/Final_version...,5


In [9]:

# Create datasets
image_dataset = ImageDataset(spatial_pos_paths, image_paths, augment_idx=0)

image_dataloader = DataLoader(image_dataset, batch_size=64, shuffle=False)


expression_dataset = ExpressionDataset(adata, sample_id, ngenes=512)
expression_dataloader = DataLoader(expression_dataset, batch_size=64, shuffle=False)
# Create DataLoaders



In [10]:

def get_image_embeddings(model,test_loader):
    test_loader = test_loader
    # model = CLIPModel().cuda()
    model.eval()

    print("Finished loading model")
    
    test_image_embeddings = []
    with torch.no_grad():
        for batch in tqdm(test_loader):
            image_features = model.encode_image(batch["image"].cuda())
            #image_embeddings = model.image_projection(image_features)
            test_image_embeddings.append(image_features)
    
    return torch.cat(test_image_embeddings)


def get_spot_embeddings(model,test_loader):
    test_loader = test_loader
    # model = CLIPModel().cuda()

    model.eval()

    print("Finished loading model")

    spot_embeddings = []
    with torch.no_grad():
        for batch in tqdm(test_loader):
            # spot_features = model.spot_encoder(batch["reduced_expression"].cuda()) 
            # spot_embeddings = model.spot_projection(spot_features)
            spot_embeddings.append(model.encode_spot(batch["reduced_expression"].cuda()))
    return torch.cat(spot_embeddings)


#2265x256, 2277x256
def find_matches(spot_embeddings, query_embeddings, top_k=1):
    #find the closest matches 
    spot_embeddings = torch.tensor(spot_embeddings)
    query_embeddings = torch.tensor(query_embeddings)
    query_embeddings = F.normalize(query_embeddings, p=2, dim=-1)
    spot_embeddings = F.normalize(spot_embeddings, p=2, dim=-1)
    dot_similarity = query_embeddings @ spot_embeddings.T   #2277x2265
    print(dot_similarity.shape)
    _, indices = torch.topk(dot_similarity.squeeze(0), k=top_k)
    
    return indices.cpu().numpy()

In [11]:

save_path = "/sibcb1/chenluonanlab8/zoujiawei/Final_version_with_foundation_model/results/BRCA/H2"

In [12]:
img_embeddings_all = get_image_embeddings(model,image_dataloader)
spot_embeddings_all = get_spot_embeddings(model,expression_dataloader)


Finished loading model


  0%|          | 0/9 [00:00<?, ?it/s]/sibcb1/chenluonanlab8/zoujiawei/miniconda3/envs/CarfHe/lib/python3.9/site-packages/torch/nn/functional.py:3060: UserWarning: Default upsampling behavior when mode=bicubic is changed to align_corners=False since 0.4.0. Please specify align_corners=True if the old behavior is desired. See the documentation of nn.Upsample for details.
  warnings.warn("Default upsampling behavior when mode={} is changed "
/sibcb1/chenluonanlab8/zoujiawei/miniconda3/envs/CarfHe/lib/python3.9/site-packages/torch/nn/functional.py:3103: UserWarning: The default behavior for interpolate/upsample with float scale_factor changed in 1.6.0 to align with other frameworks/libraries, and now uses scale_factor directly, instead of relying on the computed output size. If you wish to restore the old behavior, please set recompute_scale_factor=True. See the documentation of nn.Upsample for details. 
  warnings.warn("The default behavior for interpolate/upsample with float scale_factor

Finished loading model


100%|██████████| 10/10 [00:00<00:00, 124.68it/s]


In [13]:
#save embeddings outputs:

img_embeddings_all = img_embeddings_all.cpu().numpy()
spot_embeddings_all = spot_embeddings_all.cpu().numpy()
print(img_embeddings_all.shape)
print(spot_embeddings_all.shape)

if not os.path.exists(save_path):
    os.makedirs(save_path)


np.save(save_path + "img_embeddings_" + ".npy", img_embeddings_all.T)
np.save(save_path + "spot_embeddings_" + ".npy", spot_embeddings_all.T)


(573, 384)
(603, 384)


In [14]:

#save_path = "results/brca/"
spot_embeddings1 = np.load(save_path + "spot_embeddings_.npy")

image_embeddings1 = np.load(save_path + "img_embeddings_.npy")


In [16]:
CFG.spot_embedding

512

In [53]:
expression_gt = adata_subset.X


In [54]:
expression_gt.shape

(603, 4804)

In [55]:
adata_subset.shape

(603, 4804)

In [56]:
gene_names=adata_subset.var_names

In [57]:
#query
image_query = image_embeddings1
spot_key = spot_embeddings1
expression_key = expression_gt
#expression_gt = spot_expression1

array([[-0.52361023,  0.57831496, -0.11394548, ..., -0.13591056,
        -0.26682505,  0.64207166],
       [-1.303361  ,  0.80681086, -0.22671823, ..., -0.3583184 ,
         0.13001569, -1.5068737 ],
       [ 0.24864236,  0.40955266, -0.16418211, ..., -0.41789868,
        -0.11742353,  0.2505644 ],
       ...,
       [ 0.8657702 ,  0.23310943, -0.00581783, ..., -0.40012783,
        -1.0567788 , -0.5558361 ],
       [-1.4680324 ,  0.59530574, -0.13432063, ..., -0.31914505,
        -0.42944145,  1.2728813 ],
       [-1.3609723 ,  0.47746858, -0.08054972, ..., -0.20995276,
        -1.5101677 , -0.14856847]], dtype=float32)

In [58]:
import pandas as pd
import numpy as np

def find_matches(spot_key, image_query, top_k=1):
    """Dummy find_matches function for demonstration."""
    num_queries = image_query.shape[0]
    num_spots = spot_key.shape[0]
    # Simulate finding the closest match (replace with your actual logic)
    indices = np.random.randint(0, num_spots, size=(num_queries, top_k))
    return indices

method = "average"


if image_query.shape[1] != 384:
    image_query = image_query.T
    print("image query shape: ", image_query.shape)
if expression_gt.shape[0] != image_query.shape[0]:
    expression_gt = expression_gt.T
    print("expression_gt shape: ", expression_gt.shape)
if spot_key.shape[1] != 384:
    spot_key = spot_key.T
    print("spot_key shape: ", spot_key.shape)
if expression_key.shape[0] != spot_key.shape[0]:
    expression_key = expression_key.T
    print("expression_key shape: ", expression_key.shape)

if method == "average":
    print("finding matches, using average of top k expressions")
    indices = find_matches(spot_key, image_query, top_k=1)
    matched_spot_embeddings_pred = np.zeros((indices.shape[0], spot_key.shape[1]))
    matched_spot_expression_pred = np.zeros((indices.shape[0], expression_key.shape[1]))
    for i in range(indices.shape[0]):
        matched_spot_embeddings_pred[i,:] = np.average(spot_key[indices[i,:],:], axis=0)
        # Corrected line:  No need for average when top_k=1, directly index
        matched_spot_expression_pred[i,:] = expression_key[indices[i, 0], :]

    print("matched spot embeddings pred shape: ", matched_spot_embeddings_pred.shape)
    print("matched spot expression pred shape: ", matched_spot_expression_pred.shape)

true = expression_gt
pred = matched_spot_expression_pred
print(pred.shape)
print(true.shape)

#genewise correlation
corr = np.zeros(pred.shape[0])
for i in range(pred.shape[0]):
    corr[i] = np.corrcoef(pred[i,:], true[i,:],)[0,1]
corr = corr[~np.isnan(corr)]
print("Mean correlation across cells: ", np.mean(corr))

corr = np.zeros(pred.shape[1])
for i in range(pred.shape[1]):
    corr[i] = np.corrcoef(pred[:,i], true[:,i],)[0,1]
corr = corr[~np.isnan(corr)]
print("number of non-zero genes: ", corr.shape[0])
print("max correlation: ", np.max(corr))
print("mean correlation: ", np.mean(corr))

image query shape:  (573, 384)
expression_gt shape:  (4804, 603)
spot_key shape:  (603, 384)
finding matches, using average of top k expressions


ValueError: setting an array element with a sequence.

In [49]:
if save_path != "":
    np.save(save_path + "matched_spot_embeddings_pred.npy", matched_spot_embeddings_pred.T)
    np.save(save_path + "matched_spot_expression_pred.npy", matched_spot_expression_pred.T)

In [ ]:
indices = find_matches(spot_key, image_query, top_k=1)

In [ ]:
import pandas as pd
import torch

# Load spatial position data
spatial_pos_path = spatial_pos_paths[0]
spatial_pos_csv = pd.read_csv(spatial_pos_path, sep=",", header=None)

image_path = image_paths[0]

In [ ]:
v1 = spatial_pos_csv.iloc[1:,5]
v2 = spatial_pos_csv.iloc[1:,6]

In [ ]:
#construct heatmap of the GGC matrix
expression_gt = expression_gt
matched_spot_expression_pred_1 = np.load(save_path+"matched_spot_expression_pred.npy")


In [ ]:
save_path+"matched_spot_expression_pred.npy"

In [ ]:
gene_names = gene_names[:CFG.spot_embedding]

In [ ]:
# Convert to DataFrames
expression_gt_df = pd.DataFrame(expression_gt)
matched_spot_expression_pred_1_df = pd.DataFrame(matched_spot_expression_pred_1)

expression_data1 = pd.DataFrame(expression_gt, columns=gene_names) 
expression_data = pd.DataFrame(matched_spot_expression_pred_1.T, columns=gene_names) 


In [ ]:
import matplotlib.pyplot as plt
# 绘制相关系数的分布直方图
pearson_corr = expression_data.corrwith(expression_data1, axis=0)

# 绘制相关系数的分布直方图
plt.figure(figsize=(10, 6))
plt.hist(pearson_corr, bins=50, color='#DDA0DD', edgecolor='k', alpha=0.7)
plt.title('Distribution of Pearson Correlations between Predicted and True Gene Expression')
plt.xlabel('Pearson Correlation Coefficient')
plt.ylabel('Frequency')
plt.grid(True)

# 定义保存路径
plt.savefig(save_path + f'/pearson_correlation_histogram_2000.png')

# 显示图像
plt.show()


In [ ]:
# 选择要可视化的基因

gene_to_plot = gene_names[0]
print(gene_to_plot)
# 例如 HAL [ "HAL", "CYP3A4", "VWF", "SOX9", "KRT7", "ANXA4", "ACTA2", "DCN"] #markers from macparland paper

In [ ]:
#TLS genes
tls_genes = ['CD4', 'CD8A', 'CD74', 'CD79A', 'IL7R', 'ITGAE', 'CD1D', 'CD3D', 'CD3E', 
         'CD8B', 'CD19', 'CD22', 'CD52', 'CD79B', 'CR2', 'CXCL13', 'CXCR5', 'FCER2', 
         'MS4A1', 'PDCD1', 'PTGDS', 'TRBC2']
common_genes = np.intersect1d(gene_names, tls_genes)

# 输出结果
print(common_genes)

In [ ]:
gene_to_plot = gene_names[3]
print(gene_to_plot)
print(save_path)

In [ ]:
##image segmentation
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.colors import Normalize
import matplotlib.cm as cm

# Assume v1 and v2 are arrays or lists containing X and Y coordinates, respectively
# Assume expression_gt is a matrix where each row corresponds to gene expression levels across cells
# Assume gene_names is a list or array of gene names corresponding to the columns in expression_gt
# Assume gene_to_plot is the specific gene you want to visualize
# Assume save_path is the directory where you want to save the figure
# Assume image_path is the path to your original image
# Load the original image using OpenCV
original_image = cv2.imread(image_path)

# Convert the image from BGR (OpenCV default) to RGB
original_image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)

# Create a thumbnail by resizing the original image (adjust the scaling factor as needed)
scaling_factor = 0.1  # This scales the image down to 10% of its original size
thumbnail = cv2.resize(original_image_rgb, 
                       (int(original_image_rgb.shape[1] * scaling_factor), 
                        int(original_image_rgb.shape[0] * scaling_factor)))

# Ensure v1 and v2 are numpy arrays
v1 = np.array(v1,dtype=float)
v2 = np.array(v2,dtype=float)

v1_scaled = v1 * scaling_factor
v2_scaled = v2 * scaling_factor

# Creating a DataFrame for the coordinates and expression data
data = {
    'X': v2_scaled,  # X coordinates
    'Y': v1_scaled  # Y coordinates
}

# Convert the expression data into a DataFrame and combine it with the coordinates
plot_data1 = pd.concat([pd.DataFrame(data), expression_data1], axis=1)

# Drop any rows where the gene expression or coordinates are missing
plot_data1 = plot_data1.dropna(subset=['X', 'Y', gene_to_plot])

# Scale X and Y coordinates to match the thumbnail size
#plot_data1['X'] = plot_data1['X'] * scaling_factor
#plot_data1['Y'] = plot_data1['Y'] * scaling_factor

# Check if the gene to plot is in the DataFrame
if gene_to_plot not in plot_data1.columns:
    raise ValueError(f"{gene_to_plot} not found in expression data.")

# Set up the color mapping based on gene expression levels
vmin = np.percentile(plot_data1[gene_to_plot], 10)  # 10th percentile for lower bound
vmax = np.percentile(plot_data1[gene_to_plot], 90)  # 90th percentile for upper bound
norm = Normalize(vmin=vmin, vmax=vmax)
cmap = cm.viridis

# Create the figure and axis
fig, ax = plt.subplots(figsize=(10, 8))

# Display the thumbnail image as the background
ax.imshow(thumbnail, aspect='auto')

# Scatter plot for gene expression data on top of the thumbnail
sc = ax.scatter(plot_data1['X'], plot_data1['Y'], c=plot_data1[gene_to_plot], s=20, cmap=cmap, norm=norm, edgecolor='none')

# Add a colorbar to the plot
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Expression Level')

# Add title and labels to the plot
plt.title(f'Spatial Expression of {gene_to_plot} Groundtruth')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')

# Save the figure to the specified path
#save_path = '/sibcb1/chenluonanlab8/zoujiawei/Carhe/results/brca/A/'  # Replace with your save path
plt.savefig(save_path + f'/{gene_to_plot}_Ground.png')

# Display the plot
plt.show()

In [ ]:
##image segmentation
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.colors import Normalize
import matplotlib.cm as cm

# Assume v1 and v2 are arrays or lists containing X and Y coordinates, respectively
# Assume expression_gt is a matrix where each row corresponds to gene expression levels across cells
# Assume gene_names is a list or array of gene names corresponding to the columns in expression_gt
# Assume gene_to_plot is the specific gene you want to visualize
# Assume save_path is the directory where you want to save the figure
# Assume image_path is the path to your original image

# Create a thumbnail by resizing the original image (adjust the scaling factor as needed)
scaling_factor = 0.1  # This scales the image down to 10% of its original size
thumbnail = cv2.resize(original_image_rgb, 
                       (int(original_image_rgb.shape[1] * scaling_factor), 
                        int(original_image_rgb.shape[0] * scaling_factor)))

# Ensure v1 and v2 are numpy arrays
v1 = np.array(v1,dtype=float)
v2 = np.array(v2,dtype=float)

v1_scaled = v1 * scaling_factor
v2_scaled = v2 * scaling_factor

# Creating a DataFrame for the coordinates and expression data
data = {
    'X': v2_scaled,  # X coordinates
    'Y': v1_scaled  # Y coordinates
}

# Convert the expression data into a DataFrame and combine it with the coordinates
plot_data1 = pd.concat([pd.DataFrame(data), expression_data], axis=1)

# Drop any rows where the gene expression or coordinates are missing
plot_data1 = plot_data1.dropna(subset=['X', 'Y', gene_to_plot])

# Scale X and Y coordinates to match the thumbnail size
#plot_data1['X'] = plot_data1['X'] * scaling_factor
#plot_data1['Y'] = plot_data1['Y'] * scaling_factor

# Check if the gene to plot is in the DataFrame
if gene_to_plot not in plot_data1.columns:
    raise ValueError(f"{gene_to_plot} not found in expression data.")

# Set up the color mapping based on gene expression levels
vmin = np.percentile(plot_data1[gene_to_plot], 10)  # 10th percentile for lower bound
vmax = np.percentile(plot_data1[gene_to_plot], 90)  # 90th percentile for upper bound
norm = Normalize(vmin=vmin, vmax=vmax)
cmap = cm.viridis

# Create the figure and axis
fig, ax = plt.subplots(figsize=(10, 8))

# Display the thumbnail image as the background
ax.imshow(thumbnail, aspect='auto')

# Scatter plot for gene expression data on top of the thumbnail
sc = ax.scatter(plot_data1['X'], plot_data1['Y'], c=plot_data1[gene_to_plot], s=20, cmap=cmap, norm=norm, edgecolor='none')

# Add a colorbar to the plot
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Expression Level')

# Add title and labels to the plot
plt.title(f'Spatial Expression of {gene_to_plot} Predicted')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')

# Save the figure to the specified path
#save_path = '/sibcb1/chenluonanlab8/zoujiawei/Carhe/results/brca/A/'  # Replace with your save path
plt.savefig(save_path + f'/{gene_to_plot}_Predict.png')

# Display the plot
plt.show()

# TLS annotation

In [ ]:
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.colors import Normalize
import matplotlib.cm as cm

# 假设 expression_data 包含了所有基因的表达矩阵
# 假设 common_genes 是我们感兴趣的基因集
# 假设 v1 和 v2 是细胞的 X 和 Y 坐标

# 创建缩略图，调整图像的大小（根据需要调整缩放因子）
scaling_factor = 0.1  # 将图像缩放到原始大小的10%
thumbnail = cv2.resize(original_image_rgb, 
                       (int(original_image_rgb.shape[1] * scaling_factor), 
                        int(original_image_rgb.shape[0] * scaling_factor)))

# 确保 v1 和 v2 是 numpy 数组
v1 = np.array(v1, dtype=float)
v2 = np.array(v2, dtype=float)

# 将坐标进行缩放
v1_scaled = v1 * scaling_factor
v2_scaled = v2 * scaling_factor

# 创建包含坐标和表达数据的 DataFrame
data = {
    'X': v2_scaled,  # X 坐标
    'Y': v1_scaled,  # Y 坐标
}

# 确保这些基因存在于 expression_data 中
common_genes_in_data = [gene for gene in common_genes if gene in expression_data.columns]

# 从 expression_data 中选择这些基因的表达数据
filtered_expression_data = expression_data[common_genes_in_data]

# 计算 tls_score，这里使用简单的平均值作为 tls_score
tls_score = filtered_expression_data.mean(axis=1)

# 将 tls_score 和坐标数据合并
plot_data1 = pd.concat([pd.DataFrame(data), tls_score], axis=1)
plot_data1.columns = ['X', 'Y', 'tls_score']

# 删除缺失值
plot_data1 = plot_data1.dropna(subset=['X', 'Y', 'tls_score'])

# 设置颜色映射，基于 tls_score 计算
vmin = np.percentile(plot_data1['tls_score'], 10)  # 10th percentile for lower bound
vmax = np.percentile(plot_data1['tls_score'], 90)  # 90th percentile for upper bound
norm = Normalize(vmin=vmin, vmax=vmax)
cmap = cm.viridis  # 可以选择其他色图

# 创建图形和轴
fig, ax = plt.subplots(figsize=(10, 8))

# 显示缩略图作为背景
ax.imshow(thumbnail, aspect='auto')

# 将 tls_score 数据散点图叠加在缩略图上
sc = ax.scatter(plot_data1['X'], plot_data1['Y'], c=plot_data1['tls_score'], s=20, cmap=cmap, norm=norm, edgecolor='none')

# 添加颜色条
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('TLS Score')

# 添加标题和标签
plt.title(f'Spatial Distribution of TLS Score Predicted')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')

# 保存图像
#save_path = '/sibcb1/chenluonanlab8/zoujiawei/Carhe/results/brca/A/'  # 替换为你的保存路径
plt.savefig(save_path + f'/tls_score_Predict.png')

# 显示图像
plt.show()


In [ ]:
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.colors import Normalize
import matplotlib.cm as cm

# 假设 expression_data 包含了所有基因的表达矩阵
# 假设 common_genes 是我们感兴趣的基因集
# 假设 v1 和 v2 是细胞的 X 和 Y 坐标

# 创建缩略图，调整图像的大小（根据需要调整缩放因子）
scaling_factor = 0.1  # 将图像缩放到原始大小的10%
thumbnail = cv2.resize(original_image_rgb, 
                       (int(original_image_rgb.shape[1] * scaling_factor), 
                        int(original_image_rgb.shape[0] * scaling_factor)))

# 确保 v1 和 v2 是 numpy 数组
v1 = np.array(v1, dtype=float)
v2 = np.array(v2, dtype=float)

# 将坐标进行缩放
v1_scaled = v1 * scaling_factor
v2_scaled = v2 * scaling_factor

# 创建包含坐标和表达数据的 DataFrame
data = {
    'X': v2_scaled,  # X 坐标
    'Y': v1_scaled,  # Y 坐标
}

# 确保这些基因存在于 expression_data 中
common_genes_in_data = [gene for gene in common_genes if gene in expression_data1.columns]

# 从 expression_data 中选择这些基因的表达数据
filtered_expression_data = expression_data1[common_genes_in_data]

# 计算 tls_score，这里使用简单的平均值作为 tls_score
tls_score = filtered_expression_data.mean(axis=1)

# 将 tls_score 和坐标数据合并
plot_data1 = pd.concat([pd.DataFrame(data), tls_score], axis=1)
plot_data1.columns = ['X', 'Y', 'tls_score']

# 删除缺失值
plot_data1 = plot_data1.dropna(subset=['X', 'Y', 'tls_score'])

# 设置颜色映射，基于 tls_score 计算
vmin = np.percentile(plot_data1['tls_score'], 10)  # 10th percentile for lower bound
vmax = np.percentile(plot_data1['tls_score'], 90)  # 90th percentile for upper bound
norm = Normalize(vmin=vmin, vmax=vmax)
cmap = cm.viridis  # 可以选择其他色图

# 创建图形和轴
fig, ax = plt.subplots(figsize=(10, 8))

# 显示缩略图作为背景
ax.imshow(thumbnail, aspect='auto')

# 将 tls_score 数据散点图叠加在缩略图上
sc = ax.scatter(plot_data1['X'], plot_data1['Y'], c=plot_data1['tls_score'], s=20, cmap=cmap, norm=norm, edgecolor='none')

# 添加颜色条
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('TLS Score')

# 添加标题和标签
plt.title(f'Spatial Distribution of TLS Score Groundtruth')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')

# 保存图像
#save_path = '/sibcb1/chenluonanlab8/zoujiawei/Carhe/results/brca/A/'  # 替换为你的保存路径
plt.savefig(save_path + f'/tls_score_Ground.png')

# 显示图像
plt.show()


# spatial cluster by spagcn

In [ ]:
import os,csv,re
import pandas as pd
import numpy as np
import scanpy as sc
import math
import SpaGCN as spg
from scipy.sparse import issparse
import random, torch
import warnings
warnings.filterwarnings("ignore")
import matplotlib.colors as clr
import matplotlib.pyplot as plt
import SpaGCN as spg
#In order to read in image data, we need to install some package. Here we recommend package "opencv"
#inatll opencv in python
#!pip3 install opencv-python
import cv2

In [ ]:
import anndata as ad
import pandas as pd
v1 = v1.astype(int)
v2 = v2.astype(int)
# 假设 expression_data 是一个 pandas DataFrame，其中行表示细胞，列表示基因
# 假设 v1 和 v2 是两个数组或列表，分别表示细胞的 X 和 Y 坐标
#image_path = "/sibcb1/chenluonanlab8/zoujiawei/Carhe/brcadata/data/image_rename/H1.jpg"
# Load the original image using OpenCV
img = cv2.imread(image_path)
# 将 v1 和 v2 添加到 obs 中作为细胞的元数据
obs = pd.DataFrame(index=expression_data.index)  # 使用 expression_data 的索引作为细胞的标识
obs['X'] = v1  # X 坐标
obs['Y'] = v2 # Y 坐标

# 如果 expression_data 仅包含基因表达矩阵，则 var 需要是基因名称的 DataFrame
var = pd.DataFrame(index=expression_data.columns)  # 基因名

# 创建 AnnData 对象
adata = ad.AnnData(X=expression_data.values, obs=obs, var=var)

# 现在，你可以在 adata 中访问细胞的坐标信息
print(adata)


In [ ]:
var

In [ ]:
x_array=adata.obs["X"].tolist()
y_array=adata.obs["Y"].tolist()
x_pixel=adata.obs["X"].tolist()
y_pixel=adata.obs["Y"].tolist()

#Test coordinates on the image
img_new=img.copy()
for i in range(len(x_pixel)):
    x=x_pixel[i]
    y=y_pixel[i]
    img_new[int(x-20):int(x+20), int(y-20):int(y+20),:]=0

cv2.imwrite(save_path+'/spagcn/151673_map.jpg', img_new)

In [ ]:
#Calculate adjacent matrix
s=1
b=49
adj=spg.calculate_adj_matrix(x=x_pixel,y=y_pixel, x_pixel=x_pixel, y_pixel=y_pixel, image=img, beta=b, alpha=s, histology=True)
#If histlogy image is not available, SpaGCN can calculate the adjacent matrix using the fnction below
#adj=calculate_adj_matrix(x=x_pixel,y=y_pixel, histology=False)
np.savetxt(save_path+'/spagcn/adj.csv', adj, delimiter=',')

In [ ]:
import scanpy as sc
#adata=sc.read("./data/151673/sample_data.h5ad")
adj=np.loadtxt(save_path+'/spagcn/adj.csv', delimiter=',')
adata.var_names_make_unique()
#spg.prefilter_genes(adata,min_cells=3) # avoiding all genes are zeros
#spg.prefilter_specialgenes(adata)
#Normalize and take log for UMI
#sc.pp.normalize_per_cell(adata)
#sc.pp.log1p(adata)

In [ ]:
p=0.5 
#Find the l value given p
l=spg.search_l(p, adj, start=0.01, end=1000, tol=0.01, max_run=100)

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc

# ... your code ...

df = pd.DataFrame(adata.X)  # No need for .toarray() if adata.X is already a NumPy array.
nan_rows = df[df.isnull().any(axis=1)].index.tolist()

if nan_rows:
    print("Rows with NaN values:")
    print(nan_rows)
    # Perform imputation or removal here, potentially only on these rows.
else:
    print("No NaN values found.")


In [ ]:
#If the number of clusters known, we can use the spg.search_res() fnction to search for suitable resolution(optional)
#For this toy data, we set the number of clusters=7 since this tissue has 7 layers
n_clusters=7
#Set seed
r_seed=t_seed=n_seed=100
#Seaech for suitable resolution
res=spg.search_res(adata, adj, l, n_clusters, start=0.7, step=0.1, tol=5e-3, lr=0.05, max_epochs=20, r_seed=r_seed, t_seed=t_seed,n_seed=n_seed)

In [ ]:
clf=spg.SpaGCN()
clf.set_l(l)
#Set seed
random.seed(r_seed)
torch.manual_seed(t_seed)
np.random.seed(n_seed)
#Run
clf.train(adata,adj,init_spa=True,init="louvain",res=res, tol=5e-3, lr=0.05, max_epochs=200)
y_pred, prob=clf.predict()
adata.obs["pred"]= y_pred
adata.obs["pred"]=adata.obs["pred"].astype('category')
#Do cluster refinement(optional)
#shape="hexagon" for Visium data, "square" for ST data.
adj_2d=spg.calculate_adj_matrix(x=x_array,y=y_array, histology=False)
refined_pred=spg.refine(sample_id=adata.obs.index.tolist(), pred=adata.obs["pred"].tolist(), dis=adj_2d, shape="hexagon")
adata.obs["refined_pred"]=refined_pred
adata.obs["refined_pred"]=adata.obs["refined_pred"].astype('category')
#Save results
adata.write_h5ad(save_path+"/spagcn/results.h5ad")

In [ ]:
adata=sc.read(save_path+"/spagcn/results.h5ad")
#Set colors used
plot_color=["#F56867","#FEB915","#C798EE","#59BE86","#7495D3","#D1D1D1","#6D1A9C","#15821E","#3A84E6","#997273","#787878","#DB4C6C","#9E7A7A","#554236","#AF5F3C","#93796C","#F9BD3F","#DAB370","#877F6C","#268785"]
#Plot spatial domains
domains="pred"
num_celltype=len(adata.obs[domains].unique())
adata.uns[domains+"_colors"]=list(plot_color[:num_celltype])
ax=sc.pl.scatter(adata,alpha=1,x="X",y="Y",color=domains,title=domains,color_map=plot_color,show=False,size=100000/adata.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig(save_path+"/spagcn/pred_D.png", dpi=600)
plt.close()

#Plot refined spatial domains
domains="refined_pred"
num_celltype=len(adata.obs[domains].unique())
adata.uns[domains+"_colors"]=list(plot_color[:num_celltype])
ax=sc.pl.scatter(adata,alpha=1,x="X",y="Y",color=domains,title=domains,color_map=plot_color,show=False,size=100000/adata.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig(save_path+"/spagcn/refined_pred_D.png", dpi=600)
plt.close()

# Ground truth

In [ ]:
obs = pd.DataFrame(index=expression_data1.index)  # 使用 expression_data 的索引作为细胞的标识
obs['X'] = v2  # X 坐标
obs['Y'] = v1 # Y 坐标

# 如果 expression_data 仅包含基因表达矩阵，则 var 需要是基因名称的 DataFrame
var = pd.DataFrame(index=expression_data1.columns)  # 基因名

# 创建 AnnData 对象
adata = ad.AnnData(X=expression_data1.values, obs=obs, var=var)

# 现在，你可以在 adata 中访问细胞的坐标信息
print(adata)

In [ ]:
x_array=adata.obs["X"].tolist()
y_array=adata.obs["Y"].tolist()
x_pixel=adata.obs["X"].tolist()
y_pixel=adata.obs["Y"].tolist()

#Test coordinates on the image
img_new=img.copy()
for i in range(len(x_pixel)):
    x=x_pixel[i]
    y=y_pixel[i]
    img_new[int(x-20):int(x+20), int(y-20):int(y+20),:]=0

cv2.imwrite('./spagcn/151673_map.jpg', img_new)

In [ ]:
#Calculate adjacent matrix
s=1
b=49
adj=spg.calculate_adj_matrix(x=x_pixel,y=y_pixel, x_pixel=x_pixel, y_pixel=y_pixel, image=img, beta=b, alpha=s, histology=True)
#If histlogy image is not available, SpaGCN can calculate the adjacent matrix using the fnction below
#adj=calculate_adj_matrix(x=x_pixel,y=y_pixel, histology=False)
np.savetxt('./spagcn/adj.csv', adj, delimiter=',')

In [ ]:
#adata=sc.read("./data/151673/sample_data.h5ad")
adata.X = adata.X.astype('float64')

adj=np.loadtxt('./spagcn/adj.csv', delimiter=',')
adata.var_names_make_unique()
spg.prefilter_genes(adata,min_cells=3) # avoiding all genes are zeros
spg.prefilter_specialgenes(adata)
#Normalize and take log for UMI
sc.pp.normalize_per_cell(adata)
sc.pp.log1p(adata)

In [ ]:
p=0.5 
#Find the l value given p
l=spg.search_l(p, adj, start=0.01, end=1000, tol=0.01, max_run=100)

In [ ]:
#If the number of clusters known, we can use the spg.search_res() fnction to search for suitable resolution(optional)
#For this toy data, we set the number of clusters=7 since this tissue has 7 layers
n_clusters=7
#Set seed
r_seed=t_seed=n_seed=100
#Seaech for suitable resolution
res=spg.search_res(adata, adj, l, n_clusters, start=0.7, step=0.1, tol=5e-3, lr=0.05, max_epochs=20, r_seed=r_seed, t_seed=t_seed,n_seed=n_seed)

In [ ]:
clf=spg.SpaGCN()
clf.set_l(l)
#Set seed
random.seed(r_seed)
torch.manual_seed(t_seed)
np.random.seed(n_seed)
#Run
clf.train(adata,adj,init_spa=True,init="louvain",res=res, tol=5e-3, lr=0.05, max_epochs=200)
y_pred, prob=clf.predict()
adata.obs["pred"]= y_pred
adata.obs["pred"]=adata.obs["pred"].astype('category')
#Do cluster refinement(optional)
#shape="hexagon" for Visium data, "square" for ST data.
adj_2d=spg.calculate_adj_matrix(x=x_array,y=y_array, histology=False)
refined_pred=spg.refine(sample_id=adata.obs.index.tolist(), pred=adata.obs["pred"].tolist(), dis=adj_2d, shape="hexagon")
adata.obs["refined_pred"]=refined_pred
adata.obs["refined_pred"]=adata.obs["refined_pred"].astype('category')
#Save results
adata.write_h5ad("./spagcn/results1.h5ad")

In [ ]:
adata=sc.read("./spagcn/results1.h5ad")
#Set colors used
plot_color=["#F56867","#FEB915","#C798EE","#59BE86","#7495D3","#D1D1D1","#6D1A9C","#15821E","#3A84E6","#997273","#787878","#DB4C6C","#9E7A7A","#554236","#AF5F3C","#93796C","#F9BD3F","#DAB370","#877F6C","#268785"]
#Plot spatial domains
domains="pred"
num_celltype=len(adata.obs[domains].unique())
adata.uns[domains+"_colors"]=list(plot_color[:num_celltype])
ax=sc.pl.scatter(adata,alpha=1,x="X",y="Y",color=domains,title='ground',color_map=plot_color,show=False,size=100000/adata.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig("./spagcn/ground_D.png", dpi=600)
plt.close()

#Plot refined spatial domains
domains="refined_pred"
num_celltype=len(adata.obs[domains].unique())
adata.uns[domains+"_colors"]=list(plot_color[:num_celltype])
ax=sc.pl.scatter(adata,alpha=1,x="X",y="Y",color=domains,title='ground',color_map=plot_color,show=False,size=100000/adata.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig("./spagcn/refined_ground_D.png", dpi=600)
plt.close()

In [ ]:
adata1=sc.read("./spagcn/results.h5ad")

In [ ]:
from sklearn.metrics import adjusted_rand_score

# 计算 Adjusted Rand Index (ARI)
ari = adjusted_rand_score(adata.obs['refined_pred'], adata1.obs['refined_pred'])

print(f"ARI between adata and adata1: {ari}")


In [ ]:
from sklearn.cluster import KMeans
import numpy as np

# Assuming image_query is your data with shape (510, 384)
# Perform k-means clustering to divide the data into 7 clusters
kmeans = KMeans(n_clusters=9, random_state=9)
image_cluster = kmeans.fit_predict(image_query)

# image_cluster will contain the cluster assignment for each of the 510 samples
# For example, image_cluster[0] will be the cluster index for the first sample

# If you want to check the number of samples in each cluster
unique, counts = np.unique(image_cluster, return_counts=True)
cluster_distribution = dict(zip(unique, counts))

print("Cluster distribution:", cluster_distribution)
print("Cluster assignments for each sample:", image_cluster)


In [ ]:
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.neighbors import KernelDensity
from scipy.ndimage import gaussian_filter
from skimage.measure import find_contours

# Load the original image using OpenCV
#image_path = "/sibcb1/chenluonanlab8/zoujiawei/Carhe/brcadata/data/ST-imgs/H/H2/HE_BT24044_E1.jpg"
original_image = cv2.imread(image_path)

# Convert the image from BGR (OpenCV default) to RGB
original_image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)

# Create a thumbnail by resizing the original image (adjust the scaling factor as needed)
scaling_factor = 0.1  # This scales the image down to 10% of its original size
thumbnail = cv2.resize(original_image_rgb, 
                       (int(original_image_rgb.shape[1] * scaling_factor), 
                        int(original_image_rgb.shape[0] * scaling_factor)))

# Ensure v1 and v2 are numpy arrays of floats
v1 = np.array(v1, dtype=float)
v2 = np.array(v2, dtype=float)

# Scale down X and Y coordinates to match the thumbnail size
v1_scaled = v1 * scaling_factor
v2_scaled = v2 * scaling_factor

# Creating a DataFrame for the coordinates and cluster data
data = {
    'X': v1_scaled,  # Scaled X coordinates
    'Y': v2_scaled,  # Scaled Y coordinates
    'cluster': cluster_distribution  # Cluster assignments
}

plot_data1 = pd.DataFrame(data)

# Create the figure and axis
fig, ax = plt.subplots(figsize=(10, 8))

# Display the thumbnail image as the background
ax.imshow(thumbnail, aspect='auto')

# Draw contours around each cluster region

# Scatter plot for segmentor on top of the thumbnail
sns.scatterplot(x='X', y='Y', hue='cluster', data=plot_data1, palette='tab10', s=100, edgecolor='none', ax=ax, legend=False)

# Add title and labels to the plot
plt.title(f'Image Segmentation')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')

# Save the figure to the specified path
save_path = '/sibcb1/chenluonanlab8/zoujiawei/Carhe/results/brca/A/'  # Replace with your save path
plt.savefig(save_path + f'figures/image_seg_leiden.png', dpi=300)

# Display the plot
plt.show()


In [ ]:
save_path

In [ ]:
import scanpy as sc
import pandas as pd

# 假设expression_data的行是细胞，列是基因，如果不是，请转置
# 确保基因名和细胞名被正确设置


In [ ]:
# 基本过滤
adata = sc.AnnData(expression_data, var=pd.DataFrame(index=gene_names))
sc.pp.filter_cells(adata, min_genes=0)
sc.pp.filter_genes(adata, min_cells=0)

# 归一化每个细胞的基因表达量到相同的总数（1e4）
#sc.pp.normalize_total(adata, target_sum=1e3)

# 对数变换
#sc.pp.log1p(adata)

# 识别高变异基因
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

# 仅保留高变异基因进行后续分析
#adata = adata[:, adata.var.highly_variable]

# 数据标准化

In [ ]:
# PCA
sc.tl.pca(adata, svd_solver='arpack')

# 计算邻居图
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

# 运行t-SNE
sc.tl.tsne(adata, random_state=0)



In [ ]:
# Leiden聚类
sc.tl.leiden(adata, resolution=0.2)  # 默认值通常为1.0，你可以根据需要进行调整

# 绘制Leiden聚类的结果
sc.pl.tsne(adata, color='leiden', title='t-SNE colored by Leiden clustering')

In [ ]:
leiden_vector = adata.obs['leiden']

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# Perform Leiden clustering (if not already done)
sc.tl.leiden(adata, resolution=0.2)

# Identify the top marker genes for each Leiden cluster
sc.tl.rank_genes_groups(adata, 'leiden', method='t-test')  # You can use 'wilcoxon' or 'logreg' as well

# Extract the top 5 marker genes for each cluster
marker_genes = []
for cluster in adata.obs['leiden'].unique():
    top_genes = adata.uns['rank_genes_groups']['names'][cluster][:3]
    marker_genes.extend(top_genes)

# Remove duplicates from the list of marker genes
marker_genes = list(set(marker_genes))

# Standardize the data to mean 0 and standard deviation 1, and clip values to [-2, 2]
sc.pp.scale(adata, max_value=2)

# Define the save path and file name


# Plot the heatmap using the calculated marker genes
sc.pl.heatmap(
    adata, 
    var_names=marker_genes,  # Use the calculated marker genes
    groupby='leiden',        # Group the cells by Leiden clusters
    cmap='cividis',          # Colormap for the heatmap
    dendrogram=False,        # No dendrogram
    save='Predict_heatmap_leiden05-A.png'          # Do not save automatically
)

# Save the figure manually to the specified path
#plt.savefig(save_path + 'Predict_heatmap_leiden-B.png')


In [ ]:
leiden_vector = adata.obs['leiden']

In [ ]:
leiden_vector

In [ ]:
# 基本过滤
adata1 = sc.AnnData(expression_data1, var=pd.DataFrame(index=gene_names))
sc.pp.filter_cells(adata1, min_genes=0)
sc.pp.filter_genes(adata1, min_cells=0)

# 归一化每个细胞的基因表达量到相同的总数（1e4）
#sc.pp.normalize_total(adata1, target_sum=1e3)

# 对数变换
#adata1.X = adata1.X + np.abs(np.min(adata1.X)) + 1e-6

# Perform log transformation
#sc.pp.log1p(adata1)

# 识别高变异基因
sc.pp.highly_variable_genes(adata1, min_mean=0.0125, max_mean=3, min_disp=0.5)

# 仅保留高变异基因进行后续分析
#adata = adata[:, adata.var.highly_variable]


In [ ]:


# 再次尝试运行PCA
sc.tl.pca(adata1, svd_solver='arpack')

# 计算邻居图
sc.pp.neighbors(adata1, n_neighbors=10, n_pcs=40)

# 运行t-SNE
sc.tl.tsne(adata1, random_state=0)



In [ ]:
# Leiden聚类
sc.tl.leiden(adata1, resolution=0.2)  # 默认值通常为1.0，你可以根据需要进行调整

# 绘制Leiden聚类的结果
sc.pl.tsne(adata1, color='leiden', title='t-SNE colored by Leiden clustering',save='Ground_leiden_A.png')

In [ ]:
# Ensure leiden_vector is a Pandas Series or a list
adata1.obs['leiden_vector'] = leiden_vector
sc.pl.tsne(adata1, color='leiden_vector', title='t-SNE colored by Leiden clustering',save=False)

plt.savefig(save_path+'Ground_leiden——A.png')

In [ ]:

marker_genes = marker_genes# 替换为你的标记基因
adata1.obs['leiden_cluster'] = leiden_vector
sc.pp.scale(adata1, max_value=2) 
# 绘制热图
sc.pl.heatmap(adata1, var_names=marker_genes, groupby='leiden_cluster', cmap='cividis', dendrogram=False, save='Ground_leiden—A.png')

#plt.savefig(save_path+'Ground_leiden——byvector.png')

In [ ]:
leiden_vector1 = adata1.obs['leiden']

In [ ]:
celltype_vector = np.full(leiden_vector.shape, '', dtype=object)

# 赋值细胞类型
celltype_vector[np.isin(leiden_vector, ['0', '1'])] = 'hepatocytes-HAL'
celltype_vector[np.isin(leiden_vector, ['2', '3', '4', '7'])] = 'hepatocyte-CYP3A4'
celltype_vector[leiden_vector == '5'] = 'hepatic stellate cells'
celltype_vector[np.isin(leiden_vector, ['6', '8'])] = 'epithelial cells'

# 输出结果
print(celltype_vector)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

# 假设 v1 和 v2 是空间坐标
data = {
    'X': v1,  # X坐标
    'Y': v2,   # Y坐标
    'celltype': celltype_vector
}

# 将坐标数据创建为 DataFrame
plot_data = pd.DataFrame(data)

# 使用 KMeans 聚类
kmeans = KMeans(n_clusters=7, random_state=42)
plot_data['Cluster'] = leiden_vector
# 绘制聚类结果的图
plt.figure(figsize=(10, 8))
sns.scatterplot(x='X', y='Y', hue='celltype', data=plot_data, palette='tab10', s=50, edgecolor='none')

# 添加图像标题和坐标轴标签
plt.title('Origin celltype')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')

# 保存并显示图像
plt.savefig('figures/ground_spatial_leiden.png')
plt.show()


In [ ]:
##TLS analysis
save_path = 'results/brca/C/'
print(save_path)

##tls score by ssgeva
ori_tls = pd.read_csv('/sibcb1/chenluonanlab8/zoujiawei/Carhe/results/brca/C/ori_tls.csv')

pred_tls = pd.read_csv('/sibcb1/chenluonanlab8/zoujiawei/Carhe/results/brca/C/pred_tls.csv')
#A

In [ ]:
ori_tls  = ori_tls.iloc[0,1:]
pred_tls = pred_tls.iloc[0,1:]

In [ ]:
##image segmentation
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.colors import Normalize
import matplotlib.cm as cm

# Assume v1 and v2 are arrays or lists containing X and Y coordinates, respectively
# Assume expression_gt is a matrix where each row corresponds to gene expression levels across cells
# Assume gene_names is a list or array of gene names corresponding to the columns in expression_gt
# Assume gene_to_plot is the specific gene you want to visualize
# Assume save_path is the directory where you want to save the figure
# Assume image_path is the path to your original image
image_path = "/sibcb1/chenluonanlab8/zoujiawei/Carhe/brcadata/data/ST-imgs/H/H3/HE_BT24044_E2.jpg"
# Load the original image using OpenCV
original_image = cv2.imread(image_path)

# Convert the image from BGR (OpenCV default) to RGB
original_image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)

# Create a thumbnail by resizing the original image (adjust the scaling factor as needed)
scaling_factor = 0.1  # This scales the image down to 10% of its original size
thumbnail = cv2.resize(original_image_rgb, 
                       (int(original_image_rgb.shape[1] * scaling_factor), 
                        int(original_image_rgb.shape[0] * scaling_factor)))

# Ensure v1 and v2 are numpy arrays
v1 = np.array(v1,dtype=float)
v2 = np.array(v2,dtype=float)

v1_scaled = v1 * scaling_factor
v2_scaled = v2 * scaling_factor

# Creating a DataFrame for the coordinates and expression data
data = {
    'X': v1_scaled,  # X coordinates
    'Y': v2_scaled,
    'Ori_TLS':ori_tls,
    'Pred_TLS':pred_tls# Y coordinates
}

# Convert the expression data into a DataFrame and combine it with the coordinates
plot_data1 = pd.DataFrame(data)

# Drop any rows where the gene expression or coordinates are missing

# Scale X and Y coordinates to match the thumbnail size
#plot_data1['X'] = plot_data1['X'] * scaling_factor
#plot_data1['Y'] = plot_data1['Y'] * scaling_factor

# Set up the color mapping based on gene expression levels
vmin = np.percentile(plot_data1['Ori_TLS'], 10)  # 10th percentile for lower bound
vmax = np.percentile(plot_data1['Ori_TLS'], 90)  # 90th percentile for upper bound
norm = Normalize(vmin=vmin, vmax=vmax)
cmap = cm.viridis

# Create the figure and axis
fig, ax = plt.subplots(figsize=(10, 8))

# Display the thumbnail image as the background
ax.imshow(thumbnail, aspect='auto')

# Scatter plot for gene expression data on top of the thumbnail
scmap = ax.scatter(plot_data1['X'], plot_data1['Y'], c=plot_data1['Ori_TLS'], s=100, cmap=cmap, norm=norm, edgecolor='none')

# Add a colorbar to the plot
cbar = plt.colorbar(scmap, ax=ax)
cbar.set_label('TLS Score')

# Add title and labels to the plot
plt.title(f'Ori_TLS')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')

# Save the figure to the specified path
save_path = '/sibcb1/chenluonanlab8/zoujiawei/Carhe/results/brca/C/'  # Replace with your save path
plt.savefig(save_path + f'figures/Ori_TLS.png')

# Display the plot
plt.show()

In [ ]:
##image segmentation
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.colors import Normalize
import matplotlib.cm as cm

# Assume v1 and v2 are arrays or lists containing X and Y coordinates, respectively
# Assume expression_gt is a matrix where each row corresponds to gene expression levels across cells
# Assume gene_names is a list or array of gene names corresponding to the columns in expression_gt
# Assume gene_to_plot is the specific gene you want to visualize
# Assume save_path is the directory where you want to save the figure
# Assume image_path is the path to your original image

# Load the original image using OpenCV
original_image = cv2.imread(image_path)

# Convert the image from BGR (OpenCV default) to RGB
original_image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)

# Create a thumbnail by resizing the original image (adjust the scaling factor as needed)
scaling_factor = 0.1  # This scales the image down to 10% of its original size
thumbnail = cv2.resize(original_image_rgb, 
                       (int(original_image_rgb.shape[1] * scaling_factor), 
                        int(original_image_rgb.shape[0] * scaling_factor)))

# Ensure v1 and v2 are numpy arrays
v1 = np.array(v1,dtype=float)
v2 = np.array(v2,dtype=float)

v1_scaled = v1 * scaling_factor
v2_scaled = v2 * scaling_factor

# Creating a DataFrame for the coordinates and expression data
data = {
    'X': v1_scaled,  # X coordinates
    'Y': v2_scaled,
    'Ori_TLS':ori_tls,
    'Pred_TLS':pred_tls# Y coordinates
}

# Convert the expression data into a DataFrame and combine it with the coordinates
plot_data1 = pd.DataFrame(data)

# Drop any rows where the gene expression or coordinates are missing

# Scale X and Y coordinates to match the thumbnail size
#plot_data1['X'] = plot_data1['X'] * scaling_factor
#plot_data1['Y'] = plot_data1['Y'] * scaling_factor

# Set up the color mapping based on gene expression levels
vmin = np.percentile(plot_data1['Pred_TLS'], 10)  # 10th percentile for lower bound
vmax = np.percentile(plot_data1['Pred_TLS'], 90)  # 90th percentile for upper bound
norm = Normalize(vmin=vmin, vmax=vmax)
cmap = cm.viridis

# Create the figure and axis
fig, ax = plt.subplots(figsize=(10, 8))

# Display the thumbnail image as the background
ax.imshow(thumbnail, aspect='auto')

# Scatter plot for gene expression data on top of the thumbnail
scmap = ax.scatter(plot_data1['X'], plot_data1['Y'], c=plot_data1['Pred_TLS'], s=100, cmap=cmap, norm=norm, edgecolor='none')

# Add a colorbar to the plot
cbar = plt.colorbar(scmap, ax=ax)
cbar.set_label('TLS Score')

# Add title and labels to the plot
plt.title(f'Pred_TLS')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')

# Save the figure to the specified path
save_path = '/sibcb1/chenluonanlab8/zoujiawei/Carhe/results/brca/C/'  # Replace with your save path
plt.savefig(save_path + f'figures/Pred_TLS.png')

# Display the plot
plt.show()

In [ ]:
np.savetxt('V2.csv', v2, delimiter=',', fmt='%s')

In [ ]:
#construct heatmap of the GGC matrix
expression_gt = np.load("./data/filtered_data/3/harmony_matrix.npy")
matched_spot_expression_pred_1 = np.load("match/matched_spot_expression_pred.npy")

# matched_spot_expression_pred_2 = sc.read_h5ad('data/from_collab/harmony_C1_HisToGene_adata_pred.h5ad')
# matched_spot_expression_pred_2 = matched_spot_expression_pred_2.X.T
# matched_spot_expression_pred_3 = sc.read_h5ad('data/from_collab/harmony_C1_STNet_adata_pred.h5ad')
# matched_spot_expression_pred_3 = matched_spot_expression_pred_3.X.T

print(expression_gt.shape)
print(matched_spot_expression_pred_1.shape)


In [ ]:
image_query.shape